# RAG: 검색증강생성 완전 가이드 - 실습 코드 2: 순수 Python으로 RAG 시스템 구현 (라이브러리 없이, 초보자를 위한 상세 설명판)

- Tutorial ID: `expand-rag-fundamentals`
- Tutorial: RAG: 검색증강생성 완전 가이드
- Section ID: `expand-rag-fundamentals-code-2`
- Section: 실습 코드 2: 순수 Python으로 RAG 시스템 구현 (라이브러리 없이)

> 📘 이 노트북은 원본 실습 코드에 **개념 설명, 단계별 주석, 작은 토이(toy) 데이터 데모**를 추가한 버전입니다. RAG를 처음 공부하는 분도 각 단계에서 "지금 무슨 일이 일어나고 있는지"를 코드를 실행하지 않고 글만 읽어도 따라올 수 있도록 구성했습니다.

## 이 노트북에서 배우는 것

**RAG(Retrieval-Augmented Generation, 검색증강생성)**는 이름 그대로 "검색(Retrieval)"과 "생성(Generation)"을 결합한 방법입니다.

일반적인 LLM(거대 언어 모델)은 다음과 같은 한계를 가집니다.

- 학습 이후에 생긴 최신 정보를 모릅니다 (예: 어제 발표된 논문, 오늘 자 뉴스).
- 회사 내부 문서처럼 애초에 학습 데이터에 없었던 정보를 모릅니다.
- 모르는 내용도 그럴듯하게 지어내는 "환각(hallucination)" 현상이 생길 수 있습니다.

RAG는 이런 한계를 보완하기 위해, 질문에 바로 답하지 않고 **먼저 관련 문서를 검색해서 찾아온 뒤, 그 내용을 참고해서 답하도록** 만드는 방식입니다. 오픈북 시험에 비유하면, 모델이 답을 통째로 "암기"해서 말하는 대신 "책을 펼쳐서 확인하고" 답하게 만드는 것과 비슷합니다.

이 노트북에서는 아래 파이프라인을 외부 라이브러리 없이(numpy만 사용) 직접 구현하면서, RAG의 각 단계가 실제로 어떤 계산으로 이루어지는지 확인합니다.

```
문서 준비 → [청킹] → [임베딩] → [벡터 저장소에 저장]

사용자 질문 → [질문도 같은 방식으로 임베딩]
           → [저장소에서 가장 비슷한 청크 검색]
           → [검색 결과를 바탕으로 답변 생성]
```

실습이 끝나면, 이 단순한 구현의 각 부분이 실제 서비스에서는 어떤 라이브러리·도구로 대체되는지도 마지막 섹션에서 정리합니다.

## 코드 읽는 법

이 노트북은 정답 코드를 한 번에 실행하고 끝내는 용도가 아니라, **RAG 파이프라인이 실제 데이터 위에서 어떻게 동작하는지 한 단계씩 눈으로 확인**하기 위한 실습 노트입니다.

**읽는 순서**

1. **SimpleEmbedder** — 텍스트를 숫자 벡터로 바꾸는 부품 (TF-IDF)
2. **VectorStore** — 벡터들을 모아두고 유사도로 검색하는 부품
3. **chunk_text** — 긴 문서를 작은 조각으로 나누는 부품
4. **SimpleRAG** — 위 세 부품을 조합한 전체 파이프라인
5. **실행 예시** — 실제 문서(유명 논문 4편 요약)로 질의응답 해보기

각 부품마다 클래스·함수 정의 바로 다음에 **아주 작은 토이(toy) 데이터로 동작을 확인하는 데모 셀**을 붙여두었습니다. 결과가 왜 그렇게 나왔는지 헷갈리면, 그 위에 있는 주석을 다시 천천히 읽어보세요.

> 💡 **Tip**: 셀을 실행할 때마다 출력되는 벡터의 shape(차원), 어휘 사전의 크기, 유사도 점수를 주의 깊게 살펴보세요. 숫자 자체를 외우기보다 "입력이 어떤 모습으로 변환되어 다음 단계로 넘어가는지"를 따라가는 것이 이 노트북의 핵심 목표입니다.

## Step 0. 준비하기

RAG 시스템을 만들기 전에, 이 노트북 전체에서 사용할 라이브러리를 먼저 불러옵니다. 이번 실습은 "라이브러리 없이 RAG 구현하기"가 목표이므로, LangChain이나 임베딩 API 같은 RAG 전용 도구는 전혀 사용하지 않고 범용 도구만 사용합니다.

In [1]:
import numpy as np
# numpy는 벡터·행렬 연산을 빠르게 처리해주는 라이브러리입니다.
# RAG에서는 문서를 '숫자로 이루어진 벡터'로 표현하고, 그 벡터들 사이의
# 유사도를 계산해야 하는데, 이런 계산을 순수 파이썬 리스트로 직접 하면
# 느리고 코드도 복잡해집니다. numpy를 쓰면 벡터 연산(내적, 정규화 등)을
# 한두 줄로 간결하고 빠르게 처리할 수 있습니다.

from typing import List, Dict, Tuple
# 타입 힌트(type hint)를 쓰기 위한 도구입니다. 예를 들어 함수 시그니처가
#     def fit(self, documents: List[str])
# 처럼 되어 있으면 "documents는 문자열(str)들의 리스트(List)를 받는다"는
# 뜻입니다. 실행 결과에는 영향을 주지 않지만, 코드를 읽는 사람이 함수가
# 어떤 자료형을 주고받는지 바로 알아볼 수 있게 도와줍니다.

import json
# 결과를 파일로 저장하거나 불러올 때 사용하는 표준 라이브러리입니다.
# 이 노트북 맨 마지막의 "보너스" 셀에서 질의응답 결과를 파일로 저장할 때
# 사용할 예정입니다.

print("✅ 라이브러리 불러오기 완료")

✅ 라이브러리 불러오기 완료

## Step 1. 텍스트를 숫자로 — TF-IDF 임베딩

컴퓨터는 문자를 '의미' 그대로 이해하지 못하고, 오직 숫자 계산만 할 수 있습니다. 그래서 "이 문장과 저 문장이 얼마나 비슷한가?"를 컴퓨터가 판단하게 하려면, 먼저 문장을 **의미를 어느 정도 담은 숫자 벡터**로 바꿔야 합니다. 이 변환 과정을 **임베딩(embedding)**이라고 부릅니다.

실제 서비스(사내 챗봇, 문서 검색 등)에서는 신경망으로 학습된 임베딩 모델(OpenAI text-embedding, Cohere embed, sentence-transformers 등)을 사용합니다. 이런 모델은 "자동차"와 "차량"처럼 형태는 달라도 의미가 비슷한 단어를 가까운 벡터로 표현할 수 있습니다. 다만 원리를 이해하기 위해, 이 노트북에서는 그보다 훨씬 오래되고 단순한 고전적 방법인 **TF-IDF**를 직접 구현해봅니다.

TF-IDF는 두 부분의 곱으로 이루어집니다.

- **TF (Term Frequency, 단어 빈도)**: 한 문서 안에서 특정 단어가 얼마나 자주 등장하는가. 자주 등장할수록 그 문서의 주제를 잘 나타내는 단어일 가능성이 높다고 봅니다.
- **IDF (Inverse Document Frequency, 역문서 빈도)**: 그 단어가 전체 문서 집합에서 얼마나 "희귀한지". "the", "is"처럼 거의 모든 문서에 등장하는 단어는 어떤 문서를 다른 문서와 구별하는 데 도움이 되지 않으므로 가중치를 낮추고, 특정 문서(들)에만 등장하는 단어는 가중치를 높입니다.

즉 TF-IDF 점수가 높은 단어는 "이 문서 안에서 자주 등장하면서도, 다른 문서에서는 잘 등장하지 않는" 단어입니다. 아래 코드에서 이 두 값을 각각 어떻게 계산하는지 확인해보세요.

In [2]:
class SimpleEmbedder:
    """
    아주 단순화한 TF-IDF 임베더입니다.

    fit()으로 문서 집합을 보고 '단어 사전'과 '단어별 가중치(idf)'를 학습한 뒤,
    embed()로 개별 텍스트를 숫자 벡터로 변환합니다.
    """

    def __init__(self, max_features=5000):
        # vocab: {단어: 인덱스 번호} 형태의 '단어 사전'입니다.
        #   예) {'transformer': 0, 'attention': 1, 'model': 2, ...}
        # 이 인덱스 번호가 곧 임베딩 벡터에서 그 단어가 차지하는 '자리(위치)'가 됩니다.
        self.vocab = {}

        # idf: {단어: idf 점수} 형태의 딕셔너리. 단어별 '희귀도 가중치'를 저장합니다.
        self.idf = {}

        # max_features: 어휘 사전에 담을 단어의 최대 개수.
        # 실제 텍스트에는 수만 개의 서로 다른 단어가 등장할 수 있는데, 전부 다
        # 벡터 차원으로 쓰면 계산이 무거워지므로, 등장 빈도가 높은 상위
        # max_features개 단어만 골라서 사용합니다.
        self.max_features = max_features

    def fit(self, documents: List[str]):
        """
        여러 문서를 보고 '단어 사전(vocab)'과 '단어별 idf'를 계산합니다.
        (머신러닝에서 fit()은 관례적으로 "데이터를 보고 학습(계산)한다"는 뜻으로 씁니다.)
        """

        # doc_freq: 어떤 단어가 '전체 문서 중 몇 개의 문서'에 등장했는지 세는 딕셔너리.
        # 주의: 한 문서 안에서 같은 단어가 여러 번 나와도 그 문서에 대해서는 1번만 셉니다.
        # (예: "고양이가 고양이를 봤다"라는 문서 하나에서 '고양이'는 1로만 카운트됩니다.)
        doc_freq = {}
        total_docs = len(documents)

        for doc in documents:
            # set(...)으로 문서 안의 중복 단어를 제거해서, 문서 하나당 단어 하나는
            # 딱 1번만 세게 만듭니다.
            words = set(doc.lower().split())
            for word in words:
                doc_freq[word] = doc_freq.get(word, 0) + 1

        # 등장한 문서 수(doc_freq)가 많은 순서로 정렬한 뒤, 상위 max_features개
        # 단어만 골라 어휘 사전(vocab)을 만듭니다.
        sorted_words = sorted(doc_freq.items(), key=lambda x: -x[1])[:self.max_features]
        self.vocab = {word: i for i, (word, _) in enumerate(sorted_words)}

        # IDF(역문서빈도) 계산 공식:
        #     idf(단어) = log( 전체 문서 수 / (그 단어가 등장한 문서 수 + 1) )
        #
        # 이 공식이 의미하는 것:
        #   - 아주 많은 문서에 등장하는 단어(예: "the", "is")는 분모가 커져서 idf가
        #     0에 가까워지거나 심지어 음수가 될 수도 있습니다
        #     → "이 단어는 문서를 구별하는 데 별 도움이 안 된다"는 뜻입니다.
        #   - 소수의 문서에만 등장하는 단어(예: "transformer", "lora")는 분모가
        #     작아 idf가 커집니다 → "이 단어가 등장했다는 사실 자체가 그 문서의
        #     특징을 잘 보여준다"는 뜻입니다.
        #   - 분모에 1을 더하는 이유는 0으로 나누는 오류를 막기 위한 안전장치입니다.
        self.idf = {word: np.log(total_docs / (freq + 1)) for word, freq in doc_freq.items()}
        return self

    def embed(self, text: str) -> np.ndarray:
        """
        텍스트 한 개를 받아서, 어휘 사전 크기만큼의 길이를 가진 TF-IDF 벡터로 바꿉니다.
        예) 어휘 사전 크기가 500이면, embed()가 반환하는 벡터도 항상 길이 500입니다.
        """

        # 먼저 모두 0으로 채워진 벡터를 만들어 둡니다. (아직 아무 단어도 반영 안 된 상태)
        vec = np.zeros(len(self.vocab))

        words = text.lower().split()

        # word_count: 이 텍스트 '안에서' 각 단어가 몇 번 등장했는지 세는 딕셔너리입니다.
        # (fit()의 doc_freq와 다른 개념입니다! doc_freq는 "몇 개의 문서에 등장했는가"를
        #  세고, word_count는 "이 텍스트 하나 안에서 몇 번 등장했는가"를 셉니다.)
        word_count = {}
        for w in words:
            word_count[w] = word_count.get(w, 0) + 1

        for word, count in word_count.items():
            if word in self.vocab:
                # TF(단어빈도) = 이 텍스트에서 그 단어가 등장한 횟수 / 텍스트의 전체 단어 수
                # 텍스트가 길어질수록 같은 단어가 우연히 여러 번 나올 수 있으므로,
                # 전체 길이로 나누어 '비율'로 바꿔줍니다.
                tf = count / len(words)

                # 최종 TF-IDF 값 = TF × IDF
                # 어휘 사전에 없는 단어(학습 때 한 번도 못 본 단어)는 벡터 안에 자리
                # 자체가 없으므로 자연스럽게 무시됩니다 (if word in self.vocab 조건).
                vec[self.vocab[word]] = tf * self.idf.get(word, 1.0)

        # L2 정규화: 벡터의 길이(크기)를 1로 맞춰줍니다.
        # 이렇게 정규화해두면, 나중에 벡터끼리 내적(dot product)만 계산해도 그 값이
        # 곧바로 '코사인 유사도'가 됩니다. (다음 단계인 VectorStore에서 이 성질을 사용합니다!)
        norm = np.linalg.norm(vec)
        return vec / norm if norm > 0 else vec


print("✅ SimpleEmbedder 클래스 정의 완료")

✅ SimpleEmbedder 클래스 정의 완료

### 잠깐, 작은 예시로 확인하고 넘어가기

방금 정의한 `SimpleEmbedder`가 실제로 어떻게 동작하는지, 실제 논문 데이터로 넘어가기 전에 아주 작은 문장 3개로 먼저 확인해봅니다. 결과로 나오는 숫자들을 위 설명과 하나씩 비교해보세요.

In [3]:
toy_docs = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "cats and dogs are pets",
]

toy_embedder = SimpleEmbedder()
toy_embedder.fit(toy_docs)

print("① 어휘 사전(vocab) — {단어: 벡터 안에서의 위치}")
print(toy_embedder.vocab)
print()

print("② 단어별 IDF 점수 (소수 셋째 자리까지 반올림)")
for word, score in toy_embedder.idf.items():
    print(f"   {word:8s} : {score:.3f}")
print()
print("   → 'the', 'sat', 'on'은 문서 3개 중 2개(doc1, doc2)에 등장해서 idf가 0입니다.")
print("     '이 단어가 등장했다'는 정보가 두 문서를 구별하는 데 아무 도움이 안 된다는 뜻입니다.")
print("     반대로 'cat', 'mat', 'dog', 'log', 'cats' 등은 문서 1개에만 등장해서 idf가 더 큽니다.")
print()

vec = toy_embedder.embed(toy_docs[0])
print(f"③ 첫 번째 문서 '{toy_docs[0]}'를 임베딩한 결과")
print(np.round(vec, 3))
print(f"   벡터의 shape: {vec.shape}  ← 어휘 사전 크기와 정확히 같습니다 (총 {len(toy_embedder.vocab)}개 단어)")
print("   → 0이 아닌 값은 'cat'과 'mat' 위치뿐입니다. idf가 0인 'the'/'sat'/'on'은")
print("     아무리 자주 등장해도 최종 벡터에서는 0이 되어 버리는 것을 확인하세요.")

① 어휘 사전(vocab) — {단어: 벡터 안에서의 위치}
{'on': 0, 'the': 1, 'sat': 2, 'mat': 3, 'cat': 4, 'dog': 5, 'log': 6, 'cats': 7, 'pets': 8, 'dogs': 9, 'are': 10, 'and': 11}

② 단어별 IDF 점수 (소수 셋째 자리까지 반올림)
   mat      : 0.405
   cat      : 0.405
   on       : 0.000
   the      : 0.000
   sat      : 0.000
   dog      : 0.405
   log      : 0.405
   cats     : 0.405
   pets     : 0.405
   dogs     : 0.405
   are      : 0.405
   and      : 0.405

   → 'the', 'sat', 'on'은 문서 3개 중 2개(doc1, doc2)에 등장해서 idf가 0입니다.
     '이 단어가 등장했다'는 정보가 두 문서를 구별하는 데 아무 도움이 안 된다는 뜻입니다.
     반대로 'cat', 'mat', 'dog', 'log', 'cats' 등은 문서 1개에만 등장해서 idf가 더 큽니다.

③ 첫 번째 문서 'the cat sat on the mat'를 임베딩한 결과
[0.    0.    0.    0.707 0.707 0.    0.    0.    0.    0.    0.    0.   ]
   벡터의 shape: (12,)  ← 어휘 사전 크기와 정확히 같습니다 (총 12개 단어)
   → 0이 아닌 값은 'cat'과 'mat' 위치뿐입니다. idf가 0인 'the'/'sat'/'on'은
     아무리 자주 등장해도 최종 벡터에서는 0이 되어 버리는 것을 확인하세요.

## Step 2. 벡터 저장소와 코사인 유사도

이제 문서를 숫자 벡터로 바꾸는 방법을 만들었으니, "질문 벡터와 가장 비슷한 문서 벡터"를 찾아내는 부품이 필요합니다.

- **벡터 저장소(Vector Store)**: 임베딩 벡터들을 모아두고, 검색을 지원하는 자료구조입니다. 실제 서비스에서는 FAISS, Pinecone, Chroma, Weaviate 같은 전용 벡터 데이터베이스를 사용하지만, 여기서는 원리를 보여주기 위해 파이썬 리스트와 numpy만으로 직접 구현합니다.
- **코사인 유사도(Cosine Similarity)**: 두 벡터의 '방향'이 얼마나 비슷한지 재는 척도입니다. 벡터의 크기(길이)는 무시하고 방향만 비교하며, -1(완전히 반대 방향) ~ 1(완전히 같은 방향) 사이의 값을 가집니다. 값이 1에 가까울수록 두 텍스트가 더 비슷하다는 뜻입니다.

여기서 중요한 성질이 하나 있습니다. 앞서 `SimpleEmbedder.embed()`의 마지막 줄에서 벡터를 **L2 정규화(길이를 1로 맞춤)**했던 것을 기억하시나요? 두 벡터의 길이가 모두 1이라면, 그 둘의 **내적(dot product)을 계산한 값이 코사인 유사도와 정확히 같아집니다.** 그래서 아래 `VectorStore`는 복잡한 코사인 유사도 공식을 따로 쓰지 않고, 그냥 `np.dot`(내적)만 계산합니다.

마지막으로 **top_k**란, 유사도가 가장 높은 상위 k개의 결과만 골라내는 것을 뜻합니다.

In [4]:
class VectorStore:
    """
    임베딩 벡터들을 모아두고, 질문 벡터와 가장 비슷한 벡터를 찾아주는
    아주 단순한 '벡터 저장소'입니다.

    실제 서비스에서는 벡터가 수백만~수십억 개에 달할 수 있어서 FAISS, Pinecone,
    Chroma, Milvus 같은 전용 벡터 데이터베이스를 사용합니다. 이런 도구들은
    ANN(Approximate Nearest Neighbor, 근사 최근접 이웃) 알고리즘으로 아주 빠르게
    검색하지만, 이 노트북에서는 원리를 보여주기 위해 '저장된 모든 벡터와
    하나하나 비교하는' 가장 단순한 방식(brute-force)을 사용합니다.
    """

    def __init__(self):
        self.embeddings = []  # 각 문서(청크)의 임베딩 벡터를 저장하는 리스트
        self.documents = []   # 각 임베딩과 짝을 이루는 원본 문서(청크) 정보

    def add(self, docs: List[Dict], embedder: SimpleEmbedder):
        """문서(청크) 목록을 임베딩해서 저장소에 추가합니다."""
        for doc in docs:
            emb = embedder.embed(doc['content'])
            self.embeddings.append(emb)
            self.documents.append(doc)

        # 지금까지는 embeddings가 '벡터들의 리스트'였는데, np.array로 한 번에
        # 묶어서 2차원 행렬(문서 개수 × 어휘 사전 크기)로 만들어 둡니다. 이렇게
        # 해야 아래 search()에서 모든 문서와의 유사도를 한 번의 행렬 연산으로
        # 계산할 수 있습니다.
        self.embeddings = np.array(self.embeddings)

    def search(self, query_emb: np.ndarray, top_k: int = 3) -> List[Tuple[Dict, float]]:
        """질문 벡터(query_emb)와 가장 비슷한 상위 top_k개의 문서(청크)를 찾습니다."""

        # self.embeddings : (문서 개수, 벡터 차원) 크기의 2차원 행렬
        # query_emb       : (벡터 차원,) 크기의 1차원 벡터
        # 이 둘을 np.dot으로 곱하면, 모든 문서 벡터와 질문 벡터의 내적을 한 번에
        # 계산한 1차원 배열이 나옵니다. → shape: (문서 개수,)
        #
        # 두 벡터가 모두 L2 정규화(길이 1)되어 있으므로, 이 내적 값은 곧 코사인
        # 유사도와 같습니다.
        similarities = np.dot(self.embeddings, query_emb)

        # np.argsort(similarities)는 값을 '오름차순(작은 값 → 큰 값)'으로 정렬했을
        # 때의 원래 인덱스를 반환합니다.
        #   예) similarities = [0.1, 0.9, 0.5] → argsort 결과 = [0, 2, 1]
        #      (가장 작은 0.1의 인덱스 0이 먼저, 가장 큰 0.9의 인덱스 1이 마지막)
        #
        # 우리가 원하는 건 '유사도가 가장 높은' 상위 top_k개이므로,
        #   [-top_k:] → 오름차순 정렬 결과의 맨 뒤(=가장 큰 값들) top_k개를 자르고
        #   [::-1]    → 순서를 뒤집어서 '가장 큰 값 → 그다음 큰 값' 순서로 바꿉니다.
        top_indices = np.argsort(similarities)[-top_k:][::-1]

        # 정렬된 인덱스에 해당하는 (문서, 유사도 점수) 쌍을 리스트로 반환합니다.
        return [(self.documents[i], similarities[i]) for i in top_indices]


print("✅ VectorStore 클래스 정의 완료")

✅ VectorStore 클래스 정의 완료

### 잠깐, 작은 예시로 검색 결과 확인하기

앞서 만든 `toy_embedder`와 `toy_docs`를 그대로 이어서, 새로운 질문을 하나 던져보고 어떤 문서가 검색되는지 확인해봅니다.

In [5]:
# VectorStore.add()는 List[Dict] 형태를 기대하므로, 문자열 리스트를
# {'content': 문장} 형태의 딕셔너리 리스트로 감싸줍니다.
toy_doc_dicts = [{"content": text, "id": f"toy_{i}"} for i, text in enumerate(toy_docs)]

toy_store = VectorStore()
toy_store.add(toy_doc_dicts, toy_embedder)

toy_query = "cats and dogs"
toy_query_vec = toy_embedder.embed(toy_query)

results = toy_store.search(toy_query_vec, top_k=3)

print(f"질문: '{toy_query}'\n")
for rank, (doc, score) in enumerate(results, start=1):
    print(f"{rank}위 (유사도 {score:.3f}) : {doc['content']}")

print()
print("→ 질문 속 단어와 하나도 겹치지 않는 문서는 유사도가 정확히 0이 되는 것을 확인하세요.")
print("   ('dog'와 'dogs'도 서로 다른 단어로 취급됩니다 — 이것이 TF-IDF의 대표적인 한계입니다.)")

질문: 'cats and dogs'

1위 (유사도 0.775) : cats and dogs are pets
2위 (유사도 0.000) : the dog sat on the log
3위 (유사도 0.000) : the cat sat on the mat

→ 질문 속 단어와 하나도 겹치지 않는 문서는 유사도가 정확히 0이 되는 것을 확인하세요.
   ('dog'와 'dogs'도 서로 다른 단어로 취급됩니다 — 이것이 TF-IDF의 대표적인 한계입니다.)

## Step 3. 문서를 작은 조각으로 — 청킹(Chunking)

실제 문서(논문, 매뉴얼, 위키 페이지 등)는 수천~수만 단어에 달할 정도로 길 수 있습니다. 이렇게 긴 문서 전체를 통째로 벡터 하나로 임베딩해버리면, 문서 안의 다양한 세부 내용이 하나의 벡터에 뭉뚱그려져서 특정 질문과 관련된 부분만 정확히 찾아내기 어려워집니다. 책 한 권 전체를 문장 하나로 요약해서 색인을 만드는 것과 비슷한데, 이렇게 하면 "3장에 나온 특정 개념"을 찾고 싶어도 색인이 너무 뭉뚱그려져 있어 찾기 힘들겠죠.

그래서 RAG 시스템은 보통 문서를 작은 **청크(chunk)**로 미리 잘라서, 청크 단위로 임베딩하고 검색합니다. 이렇게 하면 훨씬 더 정밀한 검색이 가능해집니다.

**overlap(겹침)**이 필요한 이유도 있습니다. 청크 경계에서 문장이 뚝 잘리면 문맥이 끊길 수 있습니다. 예를 들어 "이 모델은 매우 효율적이다"라는 문장이 정확히 청크 경계에서 둘로 쪼개지면, 두 청크 모두 온전한 의미를 잃어버립니다. 그래서 청크끼리 일부 단어를 겹치게 만들어서, 경계 부분에서 정보가 손실되는 것을 줄입니다.

이 노트북에서는 이해를 돕기 위해 chunk_size와 overlap의 단위를 **단어 개수**로 단순화했습니다 (실제로는 토큰(token) 단위를 사용하는 경우가 더 많습니다).

In [6]:
def chunk_text(text: str, chunk_size: int = 200, overlap: int = 50) -> List[str]:
    """
    긴 텍스트를 겹치는(overlap) 작은 조각(chunk)들로 나눕니다.
    chunk_size와 overlap의 단위는 '단어 개수'입니다.
    """

    # overlap이 chunk_size보다 크거나 같으면 한 걸음(step)이 0 이하가 되어,
    # 같은 자리만 계속 반복하거나 결과가 텅 비어버리는 문제가 생깁니다.
    # 미리 조건을 확인해서 이해하기 쉬운 에러 메시지를 보여줍니다.
    assert overlap < chunk_size, "overlap은 반드시 chunk_size보다 작아야 합니다."

    words = text.split()
    chunks = []

    # step: 한 번에 몇 단어씩 건너뛰며 다음 청크를 시작할지 정하는 값입니다.
    # 예) chunk_size=10, overlap=3이면 step=7
    #     → 0번째, 7번째, 14번째, ... 단어에서 새 청크를 시작합니다.
    step = chunk_size - overlap

    for i in range(0, len(words), step):
        # words[i : i+chunk_size] → i번째 단어부터 chunk_size개를 잘라옵니다.
        # 파이썬 리스트 슬라이싱은 끝 인덱스가 리스트 길이를 넘어가도 에러 없이
        # 있는 만큼만 잘라주므로, 마지막 청크는 자연스럽게 chunk_size보다 짧아질 수 있습니다.
        chunk = ' '.join(words[i:i + chunk_size])
        if chunk.strip():  # 혹시 빈 문자열이 생기면 결과에 포함하지 않습니다.
            chunks.append(chunk)

    return chunks


print("✅ chunk_text 함수 정의 완료")

✅ chunk_text 함수 정의 완료

### 잠깐, 숫자로 만든 가짜 문서로 청킹 결과 확인하기

실제 문장으로 예시를 들면 몇 번째 단어가 어느 청크로 들어갔는지 세기 번거롭습니다. 그래서 1부터 30까지의 숫자를 공백으로 이어붙여 "단어가 30개인 가짜 문서"를 만들고, 각 숫자가 어느 청크(들)에 들어가는지 눈으로 직접 확인해봅니다.

In [7]:
sample_text = " ".join(str(n) for n in range(1, 31))  # "1 2 3 4 ... 30"
print("원본 텍스트:", sample_text)
print()

sample_chunks = chunk_text(sample_text, chunk_size=10, overlap=3)

print(f"chunk_size=10, overlap=3 (한 번에 {10 - 3}칸씩 이동) → 총 {len(sample_chunks)}개 청크 생성\n")
for i, c in enumerate(sample_chunks):
    print(f"청크 {i + 1}: {c}")

원본 텍스트: 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30

chunk_size=10, overlap=3 (한 번에 7칸씩 이동) → 총 5개 청크 생성

청크 1: 1 2 3 4 5 6 7 8 9 10
청크 2: 8 9 10 11 12 13 14 15 16 17
청크 3: 15 16 17 18 19 20 21 22 23 24
청크 4: 22 23 24 25 26 27 28 29 30
청크 5: 29 30

**관찰 1**: 연속된 청크끼리는 `overlap`(여기서는 3)만큼 단어를 서로 공유합니다. 예를 들어 청크 1의 마지막 세 숫자(`8 9 10`)와 청크 2의 처음 세 숫자를 비교해보세요.

**관찰 2**: 마지막 청크를 자세히 보면, 그 내용이 바로 앞 청크에 이미 통째로 포함되어 있는 것을 볼 수 있습니다. 문서 끝부분에서 남은 단어 수가 얼마 안 될 때, 이렇게 단순한 방식의 청킹은 거의 중복되는 작은 청크를 하나 더 만들어낼 수 있습니다 — 실제 서비스에서는 이런 경우를 감지해서 병합하거나 제거하는 후처리를 추가하기도 합니다.

## Step 4. 모든 것을 하나로 — SimpleRAG 파이프라인

지금까지 만든 세 부품(`SimpleEmbedder`, `VectorStore`, `chunk_text`)을 조합해서 실제 RAG 파이프라인을 완성합니다.

RAG라는 이름 그대로 세 단계로 이루어집니다.

- **R**etrieve (검색) — 질문과 관련된 청크를 찾아옵니다.
- **A**ugment (증강) — 찾아온 청크 내용을 하나로 합쳐서, 답을 만들 때 참고할 자료로 사용합니다.
- **G**enerate (생성) — 참고 자료를 바탕으로 답변을 만듭니다.

아래 코드에서 각 메서드가 R / A / G 중 어느 단계를 담당하는지 주석으로 표시했습니다. 다만 한 가지 짚고 넘어갈 점이 있습니다. **Generate(생성) 단계는 원래 실제 LLM(Claude, GPT 등)을 호출해서 자연스러운 문장으로 답을 받아야 하는 단계입니다.** 이 노트북은 "라이브러리·API 호출 없이 순수 파이썬만으로 RAG의 흐름을 이해하는 것"이 목표이므로, 실제 LLM 호출 대신 검색된 청크를 보기 좋게 정리해서 보여주는 방식으로 Generate 단계를 대신합니다. 코드 안 주석에 실제 서비스라면 이 부분이 어떻게 바뀌는지도 함께 적어두었습니다.

In [8]:
class SimpleRAG:
    """
    지금까지 만든 세 부품(SimpleEmbedder, VectorStore, chunk_text)을 조합해서
    완성한 RAG(Retrieval-Augmented Generation, 검색증강생성) 파이프라인입니다.
    """

    def __init__(self, chunk_size=200, chunk_overlap=50):
        self.embedder = SimpleEmbedder()
        self.vectorstore = VectorStore()
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap

    def index_documents(self, documents: List[Dict]):
        """
        문서들을 실제로 검색 가능한 상태로 만드는 '준비 작업(인덱싱)'입니다.
        질문을 받기 전에 미리 한 번만 실행해 두는 단계입니다.

        순서: 청킹 → (청크들로) 임베더 학습 → 벡터 저장소에 추가
        """

        # 1) 청킹: 각 문서를 여러 개의 작은 청크로 나눕니다.
        all_chunks = []
        for doc in documents:
            chunks = chunk_text(doc['content'], self.chunk_size, self.chunk_overlap)
            for i, chunk in enumerate(chunks):
                all_chunks.append({
                    'content': chunk,
                    'source': doc.get('id', 'unknown'),  # 이 청크가 원래 어느 문서에서 왔는지
                    'chunk_id': i,                         # 같은 문서 안에서 몇 번째 청크인지 (0부터 시작)
                    'metadata': doc.get('metadata', {})
                })

        # 2) 임베더 학습: 전체 청크 모음을 보고 어휘 사전(vocab)과 idf를 계산합니다.
        #    문서 단위가 아니라 '청크' 단위로 학습하는 이유는, 청크가 곧 검색의
        #    최소 단위이기 때문입니다.
        self.embedder.fit([c['content'] for c in all_chunks])

        # 3) 벡터 저장소에 추가: 모든 청크를 임베딩해서 저장해 둡니다.
        self.vectorstore.add(all_chunks, self.embedder)

        print(f"✅ {len(all_chunks)}개 청크 인덱싱 완료")

    def retrieve(self, query: str, top_k: int = 3) -> List[Tuple[Dict, float]]:
        """[R] Retrieve 단계 — 질문과 가장 관련 있는 청크 top_k개를 검색합니다."""

        # 질문도 문서와 '같은 벡터 공간'으로 임베딩해야 서로 비교할 수 있습니다.
        # 여기서 self.embedder는 index_documents()에서 이미 fit()이 끝난 상태이므로,
        # 학습 때 만든 어휘 사전(vocab)과 idf를 그대로 재사용합니다.
        query_emb = self.embedder.embed(query)
        return self.vectorstore.search(query_emb, top_k)

    def generate_answer(self, query: str, context_docs: List[Tuple[Dict, float]]) -> str:
        """
        [A]+[G] Augment + Generate 단계

        - Augment(증강): 검색된 여러 청크의 내용을 하나의 컨텍스트(context)
          문자열로 합칩니다.
        - Generate(생성): 원래는 이 컨텍스트 + 질문을 실제 LLM에게 프롬프트로
          전달해서 자연스러운 문장으로 된 답을 받아야 하는 단계입니다.

        ⚠️ 이 노트북은 순수 파이썬만으로 RAG의 흐름을 이해하는 것이 목표이므로,
        실제 LLM 호출 대신 검색된 청크를 보기 좋게 정리해서 보여주는 텍스트
        포매팅으로 Generate 단계를 대신합니다.

        실제 서비스라면 이 함수 안에서 대략 아래와 같은 일이 벌어집니다.

            prompt = (
                f"다음 참고 자료를 바탕으로 질문에 답하세요.\n\n"
                f"참고 자료:\n{context}\n\n질문: {query}"
            )
            answer = call_llm_api(prompt)   # 예: Claude API 호출
            return answer
        """

        # Augment: 검색된 청크들의 본문(content)만 모아 하나의 컨텍스트 문자열로 합칩니다.
        context = "\n".join([doc['content'] for doc, score in context_docs])

        # 아래부터는 실제 LLM 호출을 대신하는 '간이 생성(simplified generation)' 로직입니다.
        answer = f"질문: {query}\n\n"
        answer += f"🔍 검색된 관련 문서 ({len(context_docs)}개):\n\n"
        for i, (doc, score) in enumerate(context_docs):
            preview = doc['content'][:200]
            ellipsis = "..." if len(doc['content']) > 200 else ""
            answer += f"[문서 {i + 1}] (유사도: {score:.3f})\n"
            answer += f"출처: {doc['source']} (청크 {doc['chunk_id']})\n"
            answer += f"내용: {preview}{ellipsis}\n\n"

        return answer

    def query(self, question: str, top_k: int = 3) -> str:
        """
        RAG 전체 파이프라인을 한 번에 실행하는 '진입점' 함수입니다.
        사용하는 입장에서는 이 query() 메서드 하나만 호출하면 됩니다.
        """
        # Step 1: Retrieve — 관련 청크 검색
        context_docs = self.retrieve(question, top_k)

        # Step 2: Augment + Generate — 검색 결과를 바탕으로 답변 생성
        answer = self.generate_answer(question, context_docs)

        return answer


print("✅ SimpleRAG 클래스 정의 완료 — 이제 진짜 데이터로 실행해봅니다")

✅ SimpleRAG 클래스 정의 완료 — 이제 진짜 데이터로 실행해봅니다

## Step 5. 실전 데이터 준비하기

이제 장난감 예시를 벗어나, 유명한 딥러닝 논문 4편을 간단히 요약한 지식 베이스(knowledge base)로 실습해봅니다.

앞서 청킹을 설명할 때 "문서가 길면 여러 청크로 나뉜다"고 했는데, 이번에는 문서 하나하나를 여러 문단 분량으로 길게 작성해서, 실제로 하나의 문서가 여러 개의 청크로 나뉘는 모습을 눈으로 확인할 수 있게 했습니다.

(참고: 아래 문서 내용은 영어로 되어 있고, 뒤에서 사용할 질문들도 영어로 작성했습니다. TF-IDF는 '단어의 형태'가 정확히 일치해야 매칭이 되는 방식이라, 질문 언어와 문서 언어가 다르면 거의 검색이 되지 않기 때문입니다. 이는 이 노트북 뒷부분의 "한계점"에서 다시 다룹니다.)

In [9]:
knowledge_base = [
    {
        "id": "transformer_paper",
        "content": (
            "The Transformer architecture was introduced in the paper 'Attention Is All You Need' "
            "by Vaswani et al. in 2017. It replaced recurrent neural networks (RNNs) and convolutional "
            "neural networks (CNNs) with a mechanism called self-attention, which allows the model to "
            "weigh the importance of different words in a sequence regardless of their distance from "
            "each other. This design enables full parallelization during training, unlike RNNs, which "
            "must process tokens one at a time in sequence. The model follows an encoder-decoder "
            "structure. The encoder maps an input sequence into a set of continuous representations, "
            "and the decoder generates an output sequence one token at a time, attending to both the "
            "encoder's output and its own previously generated tokens. Each encoder and decoder layer "
            "contains a multi-head self-attention sub-layer and a position-wise feed-forward network, "
            "connected with residual connections and layer normalization. Because self-attention has no "
            "inherent notion of word order, the Transformer adds positional encodings to the input "
            "embeddings so the model can use sequence order information. The original paper's base "
            "model used 6 encoder layers, 6 decoder layers, a model dimension of 512, and 8 attention "
            "heads. This architecture became the foundation for nearly all modern large language "
            "models, including BERT and the GPT family."
        ),
    },
    {
        "id": "bert_paper",
        "content": (
            "BERT (Bidirectional Encoder Representations from Transformers) was introduced by Google "
            "in 2018. Unlike earlier language models that read text either left-to-right or "
            "right-to-left, BERT is trained to understand context from both directions at once, which "
            "is why it is called bidirectional. BERT is trained using two unsupervised tasks. The first "
            "is masked language modeling (MLM), where about 15 percent of the input tokens are randomly "
            "replaced with a special mask token, and the model must predict the original words using "
            "the surrounding context on both sides. The second task is next sentence prediction (NSP), "
            "where the model learns whether one sentence naturally follows another, which helps with "
            "tasks that require understanding relationships between sentences. After this pre-training "
            "stage, BERT can be fine-tuned on a specific downstream task, such as question answering or "
            "sentiment classification, by adding a small task-specific output layer and training on "
            "labeled data for just a few epochs. BERT-Base has 110 million parameters, 12 transformer "
            "layers, and a hidden size of 768, while BERT-Large has 340 million parameters. When it was "
            "released, BERT achieved state-of-the-art results on 11 different NLP benchmarks, including "
            "the GLUE benchmark and the SQuAD question answering dataset."
        ),
    },
    {
        "id": "gpt3_paper",
        "content": (
            "GPT-3 (Generative Pre-trained Transformer 3) was introduced by OpenAI in 2020. Its central "
            "finding was that scaling up a language model, both in parameter count and training data, "
            "allows it to perform many tasks using only a natural language prompt, without any "
            "task-specific fine-tuning. This ability is called few-shot learning, or in the extreme "
            "case of a single example, one-shot learning, and with no examples at all, zero-shot "
            "learning. The largest version of GPT-3 has 175 billion parameters, more than 100 times "
            "larger than GPT-2, and was trained on approximately 45 terabytes of text data filtered "
            "from sources including Common Crawl, WebText2, books, and Wikipedia. Like the original "
            "Transformer, GPT-3 uses a decoder-only architecture, meaning it generates text one token "
            "at a time using only previously generated tokens as context, without a separate encoder. "
            "The GPT-3 paper demonstrated that this single model, given only a short description or a "
            "handful of examples in its prompt, could perform an impressively wide range of tasks: "
            "translation between languages, answering trivia questions, writing simple code, and even "
            "basic arithmetic. This surprising generality with a fixed set of weights is often cited as "
            "a key milestone that motivated the development of later instruction-tuned and "
            "chat-oriented models such as ChatGPT."
        ),
    },
    {
        "id": "lora_paper",
        "content": (
            "LoRA (Low-Rank Adaptation) was introduced by Microsoft researchers in 2021 as an efficient "
            "way to fine-tune large pre-trained language models. Fully fine-tuning a large model means "
            "updating every one of its parameters, which requires a large amount of GPU memory and "
            "storage, especially as models grow to billions of parameters. LoRA's key idea is to freeze "
            "the original pre-trained weights entirely and instead inject small, trainable low-rank "
            "matrices into each layer of the Transformer, typically into the attention weight matrices. "
            "During fine-tuning, only these small matrices are updated, while the original weights "
            "remain unchanged. Because the rank of these injected matrices can be very small, such as 4 "
            "or 8, the number of trainable parameters can be reduced to well under 1 percent of the "
            "original model's total parameter count. This approach has several practical benefits. It "
            "dramatically reduces the memory needed for training, since gradients and optimizer states "
            "only need to be stored for the small LoRA matrices rather than the full model. It also "
            "allows multiple task-specific LoRA modules to be trained and stored separately for the "
            "same base model, and swapped in and out at inference time without duplicating the entire "
            "model. Experiments in the paper showed that LoRA can match or even exceed the performance "
            "of full fine-tuning on many tasks, despite training far fewer parameters."
        ),
    },
]

for doc in knowledge_base:
    word_count = len(doc["content"].split())
    print(f"- {doc['id']}: 약 {word_count} 단어")

- transformer_paper: 약 204 단어
- bert_paper: 약 194 단어
- gpt3_paper: 약 205 단어
- lora_paper: 약 220 단어

## Step 6. 인덱싱 실행하기

`index_documents()`를 호출하면 내부적으로 아래 순서가 실행됩니다.

1. 문서 4개 → `chunk_text()`로 각각 여러 개의 청크로 분할
2. 모든 청크를 모아서 `embedder.fit()` 실행 (어휘 사전 생성)
3. 모든 청크를 `vectorstore.add()`로 임베딩 + 저장

이번에는 청킹 효과를 눈으로 확인하기 위해, 일부러 `chunk_size`를 원래 기본값(200)보다 작은 100으로, `chunk_overlap`은 20으로 설정합니다. (실제 서비스에서는 문서 특성에 따라 훨씬 더 큰 chunk_size를 쓰기도 합니다.)

In [10]:
rag = SimpleRAG(chunk_size=100, chunk_overlap=20)
rag.index_documents(knowledge_base)

✅ 12개 청크 인덱싱 완료

### 잠깐, 실제로 몇 개의 청크가 만들어졌는지 확인하기

문서 4개가 각각 몇 개의 청크로 나뉘었는지, 그리고 각 청크의 시작 부분이 어떤 내용인지 직접 살펴봅니다.

In [11]:
current_source = None
for doc in rag.vectorstore.documents:
    if doc['source'] != current_source:
        current_source = doc['source']
        print(f"\n📄 {current_source}")
    preview = doc['content'][:70].replace("\n", " ")
    print(f"   청크 {doc['chunk_id']}: {preview}...")


📄 transformer_paper
   청크 0: The Transformer architecture was introduced in the paper 'Attention Is...
   청크 1: structure. The encoder maps an input sequence into a set of continuous...
   청크 2: can use sequence order information. The original paper's base model us...

📄 bert_paper
   청크 0: BERT (Bidirectional Encoder Representations from Transformers) was int...
   청크 1: the surrounding context on both sides. The second task is next sentenc...
   청크 2: a hidden size of 768, while BERT-Large has 340 million parameters. Whe...

📄 gpt3_paper
   청크 0: GPT-3 (Generative Pre-trained Transformer 3) was introduced by OpenAI ...
   청크 1: 100 times larger than GPT-2, and was trained on approximately 45 terab...
   청크 2: range of tasks: translation between languages, answering trivia questi...

📄 lora_paper
   청크 0: LoRA (Low-Rank Adaptation) was introduced by Microsoft researchers in ...
   청크 1: During fine-tuning, only these small matrices are updated, while the o...
   청크 2: than the full m

## Step 7. 질문하기

이제 준비가 끝났으니 실제로 질문을 던져봅니다. `rag.query()` 하나만 호출하면 내부적으로 Retrieve → Augment → Generate가 순서대로 실행됩니다.

앞서 언급했듯, 이 노트북의 검색 방식(TF-IDF)은 '단어의 형태'가 겹쳐야 검색이 되는 방식입니다. 그래서 질문도 지식 베이스와 같은 영어로 작성했습니다. (한국어로 질문하면 문서와 겹치는 단어가 전혀 없어서 의미 있는 검색 결과를 얻지 못합니다 — 궁금하다면 아래 questions 리스트에 한국어 질문을 하나 추가해서 직접 실행해보세요!)

In [12]:
questions = [
    "What is the Transformer architecture?",
    "How does LoRA reduce fine-tuning cost?",
    "What is few-shot learning?",
    "What is masked language modeling in BERT?",
]

qa_results = []  # 나중에 파일로 저장하기 위해 결과를 리스트에 모아둡니다.

print("=== RAG 질의응답 결과 ===\n")
for q in questions:
    answer = rag.query(q)
    qa_results.append({"question": q, "answer": answer})
    print(answer)
    print("=" * 60)

=== RAG 질의응답 결과 ===

질문: What is the Transformer architecture?

🔍 검색된 관련 문서 (3개):

[문서 1] (유사도: 0.128)
출처: bert_paper (청크 0)
내용: BERT (Bidirectional Encoder Representations from Transformers) was introduced by Google in 2018. Unlike earlier language models that read text either left-to-right or right-to-left, BERT is trained to...

[문서 2] (유사도: 0.080)
출처: bert_paper (청크 1)
내용: the surrounding context on both sides. The second task is next sentence prediction (NSP), where the model learns whether one sentence naturally follows another, which helps with tasks that require und...

[문서 3] (유사도: 0.079)
출처: transformer_paper (청크 0)
내용: The Transformer architecture was introduced in the paper 'Attention Is All You Need' by Vaswani et al. in 2017. It replaced recurrent neural networks (RNNs) and convolutional neural networks (CNNs) wi...


질문: How does LoRA reduce fine-tuning cost?

🔍 검색된 관련 문서 (3개):

[문서 1] (유사도: 0.252)
출처: lora_paper (청크 2)
내용: than the full model. It also allows multiple ta

### 흥미로운 관찰 하나

첫 번째 질문("What is the Transformer architecture?")의 검색 결과를 자세히 보면, 1위와 2위가 오히려 `bert_paper`이고 정작 `transformer_paper`는 3위로 밀린 것을 볼 수 있습니다. **버그가 아닙니다** — TF-IDF + 코사인 유사도 방식의 실제 한계를 보여주는 좋은 예시입니다.

질문에서 실제로 검색에 도움이 되는 단어는 사실상 'transformer'와 'architecture' 둘뿐이고, 나머지("what", "is", "the")는 거의 모든 청크에 등장하는 흔한 단어라 변별력이 없습니다 (실제로 이 코퍼스에서 'the'의 idf 값은 음수입니다!). 그런데 코사인 유사도는 질문·문서 벡터의 내적을 각 벡터 '자신의 길이'로 나누어 계산하기 때문에, 흔한 단어 비중이 높아서 벡터 길이 자체가 짧은 청크는 약간의 겹침만으로도 상대적으로 높은 점수를 받을 수 있습니다.

의미를 이해하는 신경망 기반 임베딩(예: OpenAI, Cohere 임베딩)을 쓰면 이런 문제가 훨씬 줄어듭니다 — 'Transformer'와 'architecture'가 함께 쓰인 문맥 자체를 이해하기 때문입니다. 나머지 세 질문은 의도한 문서가 정확히 1위로 검색된 것과 비교해보면, 이 현상이 "가끔" 일어나는 통계적 한계라는 것이 더 잘 느껴질 것입니다.

## 정리 — 이 실습에서 만든 것과 실제 서비스의 차이

### 우리가 직접 구현한 것

- **SimpleEmbedder**: TF-IDF 방식으로 텍스트를 벡터로 변환
- **VectorStore**: 코사인 유사도 기반 벡터 검색 (전수 비교 방식)
- **chunk_text**: 단어 개수 기준으로 겹치게 자르는 청킹
- **SimpleRAG**: 위 세 부품을 묶은 Retrieve → Augment → Generate 파이프라인

### 이 단순 구현의 한계

1. **의미가 아니라 '단어의 형태'만 비교합니다.** "cat"과 "cats", "dog"와 "dogs"조차 다른 단어로 취급되며, 동의어(예: "자동차" vs "차량")나 문맥적 의미는 전혀 이해하지 못합니다.
2. **속도가 느립니다.** 순수 파이썬 반복문 + 전수 비교 방식이라 문서가 수만~수백만 건이 되면 매 질문마다 모든 벡터를 다 비교해야 해서 비현실적으로 느려집니다.
3. **진짜 '생성(Generation)'이 아닙니다.** `generate_answer()`는 검색된 청크를 정리해서 보여줄 뿐, LLM이 자연스러운 문장으로 답을 새로 작성하지는 않습니다.

### 실제 프로덕션에서는 이렇게 대체됩니다

| 이 노트북의 구성 요소 | 실제 서비스에서 흔히 쓰는 대체재 |
| --- | --- |
| `SimpleEmbedder` (TF-IDF) | OpenAI/Cohere/Voyage AI 임베딩 API, sentence-transformers 같은 오픈소스 신경망 임베딩 모델 — 단어의 '형태'가 아니라 '의미'를 벡터로 표현 |
| `VectorStore` (전수 비교) | FAISS, Pinecone, Weaviate, Chroma, Milvus 등 전용 벡터 데이터베이스 — 대규모 벡터도 빠르게(근사) 검색 |
| `chunk_text` (단어 단위) | 토큰(token) 단위 청킹, 문장·문단 경계를 고려한 시맨틱 청킹 등 |
| `generate_answer` (텍스트 나열) | 실제 LLM API 호출 — 검색된 컨텍스트를 프롬프트에 넣어 자연어로 답변 생성 |

### 다음으로 시도해볼 만한 것

- LangChain, LlamaIndex 같은 RAG 프레임워크로 같은 파이프라인을 몇 줄로 다시 구현해보기
- `SimpleEmbedder`를 실제 임베딩 API로 바꿔서 결과가 어떻게 달라지는지 비교해보기
- `generate_answer` 안에서 실제 LLM API를 호출하도록 바꿔서 '진짜' 자연어 답변 받아보기

## 보너스 — 질의응답 결과를 파일로 저장하기

실제 서비스에서는 RAG 질의응답 로그를 나중에 분석하거나 재사용하기 위해 파일로 저장해두는 경우가 많습니다. 맨 처음 불러온 `json` 모듈을 사용해서, 방금 얻은 질문/답변 결과를 간단히 저장해봅니다.

In [13]:
with open("rag_qa_log.json", "w", encoding="utf-8") as f:
    json.dump(qa_results, f, ensure_ascii=False, indent=2)

print(f"✅ {len(qa_results)}개의 질의응답 결과를 rag_qa_log.json 파일로 저장했습니다.")

✅ 4개의 질의응답 결과를 rag_qa_log.json 파일로 저장했습니다.